# SWAP Gate Status Quo Analysis

This notebook answers the question: *what is the current state of SWAP gate usage?*

It has two sections:

**Section 1 — Dataset distribution**
- How many SWAP gates appear per circuit in the existing unitary training dataset?
- How does the SWAP count distribution look split by gate pool (circuits where 'swap' is in the pool vs. not)?

**Section 2 — Baseline model status quo**
- How many SWAPs does the trained baseline model actually use when generating circuits?
- Is the model's SWAP usage roughly calibrated to the training distribution, or does it show a strong bias?
- Does SWAP usage differ between correct and incorrect compilations?

These findings motivate the SWAP-budget conditioning experiment (training a model with `; SWAP <= N` in the prompt).

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

sys.path.append("/home/a-ldungl/projects/qcircuit-generation")

from notebooks.shared.bootstrap import setup_notebook_paths
PROJECT_ROOT = setup_notebook_paths()

In [ ]:
# -- Configuration (edit as needed) -------------------------------------------

# Training dataset to analyse (the baseline reproduction dataset)
DATASET_PATH = "./artifacts/datasets/unitary-baseline-reproduction/train"

# Eval dataset (used to sample target unitaries for model analysis)
EVAL_DATASET_PATH = "./artifacts/datasets/unitary-baseline-reproduction/eval"

# Baseline model to probe
MODEL_DIR = "./artifacts/models/unitary-baseline-reproduction/paper_unitary"

# Number of target unitaries and circuits per unitary for model analysis
NUM_EVAL_UNITARIES = 128
SAMPLES_PER_UNITARY = 64
GUIDANCE_SCALE = 7.5
SAMPLE_STEPS = 20
MAX_GATES = 12
AUTO_BATCH_SIZE = 128
EXACT_DISTANCE_TOL = 1e-8
SEED = 42

print(f"DATASET_PATH      : {DATASET_PATH}")
print(f"EVAL_DATASET_PATH : {EVAL_DATASET_PATH}")
print(f"MODEL_DIR         : {MODEL_DIR}")

## Section 1 — SWAP Distribution in the Training Dataset

In [ ]:
import hydra
from hydra.core.global_hydra import GlobalHydra
from my_genQC.utils.misc_utils import infer_torch_device
from quantum_diffusion.data.dataset import DatasetLoader

device = str(infer_torch_device())
print(f"Device: {device}")

GlobalHydra.instance().clear()
with hydra.initialize(version_base=None, config_path="../../conf"):
    cfg = hydra.compose(
        config_name="config.yaml",
        overrides=["evaluation=paper_srv"],
    )
eval_cfg = cfg["evaluation"]
eval_cfg.dataset = str(Path(DATASET_PATH).expanduser().resolve())
eval_cfg.model_dir = str(Path(MODEL_DIR).expanduser().resolve())
eval_cfg.save_output = False
eval_cfg.wandb.enable = False

loader = DatasetLoader(eval_cfg, device=device)
dataset = loader.load_dataset(eval_cfg.dataset, load_embedder=False)

print(f"\nDataset loaded: {dataset.x.shape[0]:,} circuits")
print(f"Gate pool: {dataset.gate_pool}")
print(f"Circuit tensor shape: {dataset.x.shape}")

In [ ]:
# Count SWAP gates per circuit from the integer circuit tensor.
# The vocabulary maps gate names to token IDs (1-indexed; 0 = padding).
# A SWAP gate occupies one time-step column: both qubit rows in that column
# carry |swap_id| (one positive, one negative). We count distinct time-steps
# that contain the SWAP token, i.e. any(dim=qubits) == swap_id.

gate_pool = dataset.gate_pool
vocab = {g: i + 1 for i, g in enumerate(gate_pool)}  # 1-indexed (0 = pad)
print("Vocabulary:", vocab)

swap_id = vocab.get("swap")
print(f"SWAP token ID: {swap_id}")

x = dataset.x.cpu()  # [N, qubits, time]

if swap_id is not None:
    # Count number of time steps (columns) per circuit that have any qubit == swap_id
    swap_per_circuit = (x.abs() == swap_id).any(dim=1).sum(dim=1).numpy()  # [N]
else:
    swap_per_circuit = np.zeros(x.shape[0], dtype=np.int64)
    print("WARNING: 'swap' not in gate pool — all counts will be 0")

print(f"\nTotal circuits: {len(swap_per_circuit):,}")
print(f"Circuits with ≥1 SWAP : {(swap_per_circuit > 0).sum():,}  ({(swap_per_circuit > 0).mean()*100:.1f}%)")
print(f"Circuits with 0 SWAPs : {(swap_per_circuit == 0).sum():,}  ({(swap_per_circuit == 0).mean()*100:.1f}%)")
print(f"SWAP count statistics  : mean={swap_per_circuit.mean():.3f}, median={np.median(swap_per_circuit):.1f}, max={swap_per_circuit.max()}")

In [ ]:
# Split: circuits where 'swap' is mentioned in the label prompt vs. not
labels = np.array([str(y) for y in dataset.y])
has_swap_in_pool = np.array(["'swap'" in lbl or '"swap"' in lbl for lbl in labels])

swap_in_pool  = swap_per_circuit[has_swap_in_pool]
swap_not_pool = swap_per_circuit[~has_swap_in_pool]

print(f"Circuits with 'swap' in gate pool : {has_swap_in_pool.sum():,}  ({has_swap_in_pool.mean()*100:.1f}%)")
print(f"Circuits without 'swap' in pool   : {(~has_swap_in_pool).sum():,}  ({(~has_swap_in_pool).mean()*100:.1f}%)")
print()
if len(swap_in_pool):
    print(f"SWAP in pool  — mean: {swap_in_pool.mean():.3f}, median: {np.median(swap_in_pool):.1f}, "
          f"zero-SWAP fraction: {(swap_in_pool==0).mean()*100:.1f}%")
if len(swap_not_pool):
    print(f"SWAP not pool — mean: {swap_not_pool.mean():.3f}, median: {np.median(swap_not_pool):.1f}, "
          f"zero-SWAP fraction: {(swap_not_pool==0).mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=130)

max_swap = int(swap_per_circuit.max()) + 1
bins = np.arange(-0.5, max_swap + 0.5, 1)

# All circuits
axes[0].hist(swap_per_circuit, bins=bins, color="#5f8fcb", edgecolor="white")
axes[0].set_title("All circuits")
axes[0].set_xlabel("SWAP gates per circuit")
axes[0].set_ylabel("Count")
axes[0].grid(True, alpha=0.25, axis="y")

# Only where 'swap' is in the gate pool
if len(swap_in_pool):
    axes[1].hist(swap_in_pool, bins=bins, color="#63b17d", edgecolor="white")
axes[1].set_title("'swap' in gate pool")
axes[1].set_xlabel("SWAP gates per circuit")
axes[1].grid(True, alpha=0.25, axis="y")

# CDF for 'swap in pool' — useful to see how many circuits satisfy SWAP <= k
if len(swap_in_pool):
    sorted_counts = np.sort(swap_in_pool)
    cdf = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
    axes[2].step(sorted_counts, cdf * 100, color="#eb7d3c", linewidth=2)
    axes[2].set_xlabel("SWAP budget k")
    axes[2].set_ylabel("% circuits with SWAP ≤ k")
    axes[2].set_title("CDF of SWAP count (swap in pool)")
    axes[2].grid(True, alpha=0.25)
    for k in [0, 1, 2, 3]:
        pct = (swap_in_pool <= k).mean() * 100
        axes[2].annotate(f"k={k}: {pct:.1f}%", xy=(k, pct),
                         xytext=(k + 0.2, pct - 5), fontsize=8,
                         arrowprops=dict(arrowstyle="->", lw=0.8))

fig.suptitle("Training Dataset: SWAP Gate Distribution", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Tabular breakdown
rows = []
for k in range(max_swap):
    rows.append({
        "swap_count": k,
        "count_all": int((swap_per_circuit == k).sum()),
        "frac_all": float((swap_per_circuit == k).mean()),
        "count_in_pool": int((swap_in_pool == k).sum()) if len(swap_in_pool) else 0,
        "frac_in_pool": float((swap_in_pool == k).mean()) if len(swap_in_pool) else float("nan"),
    })
dist_df = pd.DataFrame(rows)
display(dist_df)

## Section 2 — Baseline Model Status Quo

Generate circuits with the trained baseline model and measure actual SWAP usage.

In [ ]:
import random

from my_genQC.inference.sampling import generate_compilation_tensors, decode_tensors_to_backend
from my_genQC.inference.evaluation_helper import get_unitaries
from my_genQC.inference.eval_metrics import UnitaryInfidelityNorm
from my_genQC.pipeline.diffusion_pipeline import DiffusionPipeline
from my_genQC.platform.simulation import CircuitBackendType, Simulator
from my_genQC.platform.tokenizer.circuits_tokenizer import CircuitTokenizer

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Load eval dataset (contains target unitaries)
GlobalHydra.instance().clear()
with hydra.initialize(version_base=None, config_path="../../conf"):
    cfg2 = hydra.compose(config_name="config.yaml", overrides=["evaluation=paper_srv"])
eval_cfg2 = cfg2["evaluation"]
eval_cfg2.dataset = str(Path(EVAL_DATASET_PATH).expanduser().resolve())
eval_cfg2.model_dir = str(Path(MODEL_DIR).expanduser().resolve())
eval_cfg2.save_output = False
eval_cfg2.wandb.enable = False

eval_loader = DatasetLoader(eval_cfg2, device=device)
eval_dataset = eval_loader.load_dataset(eval_cfg2.dataset, load_embedder=False)
print(f"Eval dataset: {len(eval_dataset.y):,} circuits, gate pool: {eval_dataset.gate_pool}")

# Load pipeline
model_path = Path(MODEL_DIR).expanduser().resolve()
pipeline = DiffusionPipeline.from_config_file(
    config_path=str(model_path) + "/", device=device
)
pipeline.guidance_sample_mode = "rescaled"
pipeline.scheduler.set_timesteps(SAMPLE_STEPS)

gate_pool = getattr(pipeline, "gate_pool", None) or eval_dataset.gate_pool
vocabulary = {gate: idx for idx, gate in enumerate(gate_pool)}
tokenizer = CircuitTokenizer(vocabulary)
simulator = Simulator(CircuitBackendType.QISKIT)

print(f"Pipeline gate pool: {gate_pool}")

In [ ]:
# Sample target unitary indices
rng = random.Random(SEED)
eval_indices = rng.sample(range(len(eval_dataset.y)), k=min(NUM_EVAL_UNITARIES, len(eval_dataset.y)))

# Full gate pool prompt (no SWAP budget — this is the status quo)
full_pool_prompt = f"Compile using: {[str(g) for g in gate_pool]}"
print(f"Prompt used: {full_pool_prompt}")

infidelity_fn = UnitaryInfidelityNorm()

rows = []

for idx in tqdm(eval_indices, desc="Evaluating baseline model"):
    target_u_split = eval_dataset.U[idx]  # [2, dim, dim]
    target_u = target_u_split[0].numpy() + 1j * target_u_split[1].numpy()

    tensors_out = generate_compilation_tensors(
        pipeline=pipeline,
        prompt=[full_pool_prompt],
        U=target_u_split.unsqueeze(0),
        samples=SAMPLES_PER_UNITARY,
        system_size=eval_dataset.x.shape[1],
        num_of_qubits=getattr(eval_dataset.params_config, "num_of_qubits", eval_dataset.x.shape[1]),
        max_gates=MAX_GATES,
        g=GUIDANCE_SCALE,
        auto_batch_size=AUTO_BATCH_SIZE,
        enable_params=False,
        no_bar=True,
    )

    decoded_circuits, _ = decode_tensors_to_backend(
        simulator=simulator,
        tokenizer=tokenizer,
        tensors=tensors_out,
        params=None,
        silent=True,
        n_jobs=1,
        filter_errs=False,
    )

    for qc in decoded_circuits:
        if qc is None:
            rows.append({"dataset_idx": idx, "swap_count": None, "exact": False, "valid": False})
            continue
        swap_count = sum(1 for inst in qc.data if inst.operation.name == "swap")
        gen_u = simulator.backend.get_unitary(qc)
        dist = float(UnitaryInfidelityNorm.distance(
            torch.as_tensor(gen_u), torch.as_tensor(target_u)
        ).item())
        exact = dist <= EXACT_DISTANCE_TOL
        rows.append({"dataset_idx": idx, "swap_count": swap_count, "exact": exact, "valid": True})

gen_df = pd.DataFrame(rows)
valid_df = gen_df[gen_df["valid"]].copy()
print(f"\nGenerated {len(gen_df):,} circuits — valid: {len(valid_df):,} ({len(valid_df)/len(gen_df)*100:.1f}%)")
print(f"Exact compilations: {valid_df['exact'].sum():,} ({valid_df['exact'].mean()*100:.1f}%)")

In [ ]:
# SWAP distribution in generated circuits
gen_swaps = valid_df["swap_count"].dropna().astype(int)
exact_swaps = valid_df[valid_df["exact"]]["swap_count"].dropna().astype(int)
wrong_swaps = valid_df[~valid_df["exact"]]["swap_count"].dropna().astype(int)

print("Generated circuits (all valid):")
print(f"  mean SWAP: {gen_swaps.mean():.3f}, median: {gen_swaps.median():.1f}")
print(f"  zero-SWAP fraction: {(gen_swaps == 0).mean()*100:.1f}%")
if len(exact_swaps):
    print(f"\nCorrect compilations:")
    print(f"  mean SWAP: {exact_swaps.mean():.3f}, median: {exact_swaps.median():.1f}")
    print(f"  zero-SWAP fraction: {(exact_swaps == 0).mean()*100:.1f}%")
if len(wrong_swaps):
    print(f"\nWrong compilations:")
    print(f"  mean SWAP: {wrong_swaps.mean():.3f}, median: {wrong_swaps.median():.1f}")
    print(f"  zero-SWAP fraction: {(wrong_swaps == 0).mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=130)

all_max = max(
    swap_in_pool.max() if len(swap_in_pool) else 0,
    gen_swaps.max() if len(gen_swaps) else 0,
)
bins = np.arange(-0.5, all_max + 1.5, 1)

# Dataset vs. generated: normalised to fraction for fair comparison
axes[0].hist(swap_in_pool, bins=bins,
             weights=np.ones(len(swap_in_pool)) / len(swap_in_pool),
             alpha=0.6, color="#5f8fcb", label="dataset (swap in pool)")
if len(gen_swaps):
    axes[0].hist(gen_swaps, bins=bins,
                 weights=np.ones(len(gen_swaps)) / len(gen_swaps),
                 alpha=0.6, color="#eb7d3c", label="generated")
axes[0].set_xlabel("SWAP gates per circuit")
axes[0].set_ylabel("Fraction")
axes[0].set_title("Dataset vs. Generated (normalised)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.25, axis="y")

# Correct vs. wrong compilations
if len(exact_swaps) and len(wrong_swaps):
    axes[1].hist(exact_swaps, bins=bins,
                 weights=np.ones(len(exact_swaps)) / len(exact_swaps),
                 alpha=0.65, color="#63b17d", label=f"correct ({len(exact_swaps):,})")
    axes[1].hist(wrong_swaps, bins=bins,
                 weights=np.ones(len(wrong_swaps)) / len(wrong_swaps),
                 alpha=0.65, color="#e05a5a", label=f"wrong ({len(wrong_swaps):,})")
    axes[1].set_xlabel("SWAP gates per circuit")
    axes[1].set_ylabel("Fraction")
    axes[1].set_title("Correct vs. Wrong compilations")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.25, axis="y")

# Average SWAP count per requested compilation (per unitary stats)
per_unitary_swap = valid_df.groupby("dataset_idx")["swap_count"].mean().rename("mean_swap")
axes[2].hist(per_unitary_swap, bins=20, color="#9b6bb5", edgecolor="white")
axes[2].set_xlabel("Mean SWAP count across 64 samples")
axes[2].set_ylabel("Unitaries")
axes[2].set_title("Per-unitary mean SWAP count")
axes[2].grid(True, alpha=0.25, axis="y")

fig.suptitle("Baseline Model: SWAP Gate Usage", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
summary_rows = []
for k in range(int(gen_swaps.max()) + 2 if len(gen_swaps) else 1):
    summary_rows.append({
        "swap_count": k,
        "dataset_frac": float((swap_in_pool == k).mean()) if len(swap_in_pool) else float("nan"),
        "generated_frac": float((gen_swaps == k).mean()) if len(gen_swaps) else float("nan"),
        "correct_frac": float((exact_swaps == k).mean()) if len(exact_swaps) else float("nan"),
        "wrong_frac": float((wrong_swaps == k).mean()) if len(wrong_swaps) else float("nan"),
    })
summary_df = pd.DataFrame(summary_rows)
print("SWAP count distribution (fractions):")
display(summary_df.round(4))

## Conclusion

The cells above establish the baseline. Key things to look for:

1. **Is the model roughly calibrated?** Compare `dataset_frac` vs `generated_frac` in the summary table. If they are similar, the model learns the dataset's SWAP distribution well. If generated SWAP counts are systematically higher or lower, the model has a bias.

2. **Do correct compilations use fewer SWAPs?** If `correct_frac` has more mass at 0 than `wrong_frac`, reducing SWAPs might actually help compilation quality — a nice motivation for the SWAP budget.

3. **Is there enough SWAP variation?** The CDF plot (Section 1) shows what fraction of training circuits satisfy `SWAP <= k` for each k. If ~30–50% of circuits have `SWAP <= 1`, the model has plenty of examples to learn low-SWAP behaviour from the budget prompt.

**Next step:** Train `unitary_swap_constrained` (same architecture as baseline, dataset labelled with `; SWAP <= N`), then evaluate with `swap_constrained_evaluation.ipynb`.